# Домашнее задание №1

**Студент:** Леонтьев Максим

**Выполнены задания:** 1, 2, 3

## Реализация классов банковских счетов

### Импорты

In [52]:
from datetime import datetime
import re
import math
import csv
import json
import os

### Базовый класс Account

In [53]:
class Account:
    """Базовый класс банковского счёта.

    Хранит баланс и историю операций, реализует операции пополнения и снятия.
    Номер счёта присваивается автоматически в формате 'ACC-XXXX'.
    """
    _account_counter: int = 1000  # общий счётчик для всех созданных счетов
    account_type: str = "base"

    def __init__(self, account_holder: str, balance: float = 0.0) -> None:
        """Инициализирует банковский счёт.

        :param account_holder: Имя владельца в формате «Имя Фамилия»
               (кириллицей или латиницей, с заглавных букв).
        :param balance: Начальный баланс (по умолчанию 0).
        :raises ValueError: Если имя не соответствует формату или баланс отрицательный.
        """
        if not self._is_valid_holder_name(account_holder):
            raise ValueError("Имя владельца счёта должно быть в формате "
                             "«Имя Фамилия» с заглавных букв, кириллицей или латиницей")

        if not self._is_valid_number(balance, allow_zero=True):
            raise ValueError("Начальный баланс не может быть отрицательным")

        # Номер формируется автоматически: ACC-1000, ACC-1001, ...
        # Счётчик меняется через Account, а не self, чтобы он был общим
        # для базового класса и наследников
        self.account_number: str = f"ACC-{Account._account_counter}"
        Account._account_counter += 1

        self.holder: str = account_holder.strip()
        self._balance: float = float(balance)
        self.operations_history: list[dict] = []

    def deposit(self, amount: float) -> None:
        """Пополняет счёт и записывает операцию в историю.

        :param amount: Сумма пополнения, должна быть положительной.
        :raises ValueError: Если сумма не является положительным числом.
        """
        if not self._is_valid_number(amount):
            raise ValueError("Сумма пополнения должна быть положительной")

        self._balance += amount
        self._log_operation("deposit", amount, "success")

    def withdraw(self, amount: float) -> bool:
        """Снимает деньги со счёта.

        Если средств недостаточно, баланс не меняется, неудачная
        попытка фиксируется в истории со статусом 'fail'.

        :param amount: Сумма снятия, должна быть положительной.
        :return: True при успешном снятии, False при нехватке средств.
        :raises ValueError: Если сумма не является положительным числом.
        """
        if not self._is_valid_number(amount):
            raise ValueError("Сумма снятия должна быть положительной")

        if amount > self._balance:
            self._log_operation("withdraw", amount, "fail")
            return False

        self._balance -= amount
        self._log_operation("withdraw", amount, "success")
        return True

    def get_balance(self) -> float:
        """Возвращает текущий баланс счёта.

        :return: Текущий баланс.
        """
        return self._balance

    def get_history(self) -> str:
        """Возвращает историю операций в читаемом виде.

        Операции отсортированы от новых к старым.

        :return: Отформатированная строка с историей операций
                 или сообщение о том, что история пуста.
        """
        return self._format_operations(self.operations_history[::-1])

    def get_largest_operations(self, n: int = 5) -> str:
        """Возвращает n крупных успешных операций.

        Операции отсортированы по убыванию суммы; при равных суммах –
        по убыванию даты.
        Операции со статусом 'fail' не включаются.

        :param n: Количество операций для вывода (по умолчанию 5).
        :return: Отформатированная строка с n крупными операциями.
        :raises ValueError: Если n не является положительным целым числом.
        """
        if not isinstance(n, int) or isinstance(n, bool) or n <= 0:
            raise ValueError("Параметр n должен быть положительным целым числом")

        successful_operations = [
            operation
            for operation in self.operations_history
            if operation["status"] == "success"
        ]

        # Сортировка по убыванию суммы, при равных суммах — более новая дата выше
        sorted_operations = sorted(
            successful_operations,
            key=lambda op: (op["amount"], op["datetime"]),
            reverse=True,
        )[:n]

        return self._format_operations(sorted_operations)

    def clean_history(self, raw_transactions: list[dict]) -> list[dict]:
        """Очищает список "грязных" транзакций.

        Транзакция считается невалидной и отбрасывается в случаях, если
        тип операции неизвестен, статус не 'success'/'fail',
        сумма не является положительным числом,
        дата не соответствует поддерживаемым форматам,
        баланс после операции не является неотрицательным числом.
        Также невалидными считаются операции, у которых отсутствует любое поле.

        :param raw_transactions: Список словарей с ключами 'account_number',
                                 'account_type', 'date', 'operation',
                                 'amount', 'balance_after', 'status'.
        :return: Список очищенных записей в формате operations_history.
        """
        cleaned = []

        valid_types = {"deposit", "withdraw"}
        if self.account_type == "savings":
            valid_types.add("interest")

        valid_statuses = {"success", "fail"}
        date_formats = ["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M"]

        for operation in raw_transactions:
            try:
                operation_type = str(operation["operation"]).strip().lower()
                if operation_type not in valid_types:
                    continue

                status = str(operation["status"]).strip().lower()
                if status not in valid_statuses:
                    continue

                amount = float(operation["amount"])
                if not self._is_valid_number(amount):
                    continue

                date_time_str = str(operation["date"]).strip()
                date_time_obj = None
                for date_format in date_formats:
                    try:
                        date_time_obj = datetime.strptime(date_time_str, date_format)
                        break
                    except ValueError:
                        continue

                if date_time_obj is None:
                    continue

                balance = float(operation["balance_after"])
                if not self._is_valid_number(balance, allow_zero=True):
                    continue

                cleaned.append({
                    "type": operation_type,
                    "amount": amount,
                    "datetime": date_time_obj,
                    "balance": balance,
                    "status": status,
                })
            # Отсутствующий ключ, нечисловое значение или другая ошибка формата
            # означают, что запись невалидна и её нужно пропустить
            except (KeyError, ValueError, TypeError):
                continue

        return cleaned

    def load_history(self, file_path: str) -> None:
        """Загружает и добавляет в историю счёта операции из файла.

        Поддерживаются форматы .csv и .json. Файл общий для всех счетов,
        поэтому выбираются записи с account_number, совпадающим с этим счётом.
        Выбранные операции валидируются через clean_history() перед добавлением.
        После загрузки история сортируется по дате, а баланс счёта обновляется
        значением баланса из последней по времени успешной операции.

        :param file_path: Путь к файлу с транзакциями (.csv или .json).
        :raises FileNotFoundError: Если файл не найден.
        :raises ValueError: Если расширение файла не .csv и не .json.
        """
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Файл {file_path} не найден")

        raw_operations: list[dict] = []

        if file_path.endswith(".json"):
            with open(file_path, "r", encoding="utf-8") as f:
                raw_operations = json.load(f)
        elif file_path.endswith(".csv"):
            with open(file_path, "r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                raw_operations = list(reader)
        else:
            raise ValueError("Поддерживаются только форматы .csv и .json")

        # Файл общий для всех счетов
        # выбираем операции этого счёта
        account_operations = [
            operation for operation in raw_operations
            if operation.get("account_number") == self.account_number
        ]

        valid_operations = self.clean_history(account_operations)

        if valid_operations:
            self.operations_history.extend(valid_operations)
            self.operations_history.sort(key=lambda op: op["datetime"])

            # Баланс берём из последней по времени успешной
            # операции во всей истории
            last_successful = None
            for operation in reversed(self.operations_history):
                if operation["status"] == "success":
                    last_successful = operation
                    break

            if last_successful is not None:
                self._balance = last_successful["balance"]

    def _log_operation(
            self, operation_type: str, amount: float, status: str
    ) -> None:
        """Добавляет данные операции в историю.

        :param operation_type: Тип операции ('deposit', 'withdraw', 'interest').
        :param amount: Сумма операции.
        :param status: Статус операции ('success' или 'fail').
        """
        self.operations_history.append({
            "type": operation_type,
            "amount": amount,
            "datetime": datetime.now(),
            "balance": self._balance,
            "status": status,
        })

    @staticmethod
    def _is_valid_holder_name(name: str) -> bool:
        """Проверяет формат имени владельца счёта.

        Формат: «Имя Фамилия» с заглавных букв
        кириллицей или латиницей.

        :param name: Проверяемое имя.
        :return: True, если имя соответствует формату, иначе False.
        """
        if not isinstance(name, str):
            return False
        pattern = r"^([А-ЯЁ][а-яё]+ [А-ЯЁ][а-яё]+|[A-Z][a-z]+ [A-Z][a-z]+)$"
        return re.match(pattern, name.strip()) is not None

    @staticmethod
    def _is_valid_number(value: float, allow_zero: bool = False) -> bool:
        """Проверяет, что value — конечное число нужного знака.

        :param value: Проверяемое значение.
        :param allow_zero: Если True, допускается value >= 0,
                           иначе требуется value > 0 (по умолчанию False).
        :return: True, если значение валидно, иначе False.
        """
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            return False
        if not math.isfinite(value):
            return False
        if allow_zero:
            return value >= 0
        return value > 0

    @staticmethod
    def _format_operations(operations: list[dict]) -> str:
        """Форматирует список операций в читаемую многострочную строку.

        :param operations: Список операций.
        :return: Отформатированная строка или сообщение о пустой истории.
        """
        if not operations:
            return "История операций пуста"

        lines = []
        for operation in operations:
            date_time = operation["datetime"].strftime("%Y-%m-%d %H:%M:%S")
            lines.append(
                f"[{date_time}] {operation['type'].upper()}: "
                f"{operation['amount']:.2f}, Баланс: {operation['balance']:.2f}, "
                f"Статус: {operation['status']}"
            )
        return "\n".join(lines)

### Класс CheckingAccount (расчётный счёт)

In [54]:
class CheckingAccount(Account):
    """Расчётный счёт: логика не отличается от базового Account."""
    account_type: str = "checking"

### Класс SavingsAccount (сберегательный счёт)

In [55]:
class SavingsAccount(Account):
    """Сберегательный счёт: начисляет проценты и ограничивает снятие.

    Снять можно не более 50% от текущего баланса за одну операцию.
    """
    account_type: str = "savings"

    def apply_interest(self, rate: float) -> None:
        """Начисляет проценты на остаток счёта.

        Операция записывается в историю с типом 'interest'.

        :param rate: Ставка в виде доли (0.07 = 7%),
                     должна быть положительным числом.
        :raises ValueError: Если ставка не является положительным числом.
        """
        if not self._is_valid_number(rate):
            raise ValueError("Процентная ставка должна быть положительным числом")

        if self._balance <= 0:
            return

        interest = self._balance * rate
        self._balance += interest
        self._log_operation("interest", interest, "success")

    def withdraw(self, amount: float) -> bool:
        """Снимает деньги со счёта с ограничением в 50% от баланса.

        При превышении лимита в 50% операция фиксируется в истории со
        статусом 'fail'; иначе списание выполняется через базовый Account.withdraw.

        :param amount: Сумма снятия, должна быть положительной и не
                       превышать 50% от текущего баланса.
        :return: True при успешном снятии, False при превышении лимита.
        :raises ValueError: Если сумма не является положительным числом.
        """
        if not self._is_valid_number(amount):
            raise ValueError("Сумма снятия должна быть положительной")

        if amount > self._balance * 0.5:
            self._log_operation("withdraw", amount, "fail")
            return False

        return super().withdraw(amount)

## Демонстрация работы

### Операции класса `Account`

In [56]:
print("Создание счетов и проверка нумерации")

acc1 = Account("Иван Иванов", 1000)
acc2 = Account("John Smith")

print(f"Счёт 1: {acc1.account_number}, Владелец: {acc1.holder}, Баланс: {acc1.get_balance():.2f}")
print(f"Счёт 2: {acc2.account_number}, Владелец: {acc2.holder}, Баланс: {acc2.get_balance():.2f}")

print("\nПополнение и успешное снятие")

acc1.deposit(500)
print(f"Баланс {acc1.account_number} после deposit(500): {acc1.get_balance():.2f}")

acc1.withdraw(300)
print(f"Баланс {acc1.account_number} после withdraw(300): {acc1.get_balance():.2f}")

print("\nСнятие, если недостаточно средств на счёте")
result = acc1.withdraw(5000)
print(f"Результат снятия: {result}, баланс {acc1.account_number}: {acc1.get_balance():.2f}")

print(f"\nИстория операций счёта {acc1.account_number}")
print(acc1.get_history())

Создание счетов и проверка нумерации
Счёт 1: ACC-1000, Владелец: Иван Иванов, Баланс: 1000.00
Счёт 2: ACC-1001, Владелец: John Smith, Баланс: 0.00

Пополнение и успешное снятие
Баланс ACC-1000 после deposit(500): 1500.00
Баланс ACC-1000 после withdraw(300): 1200.00

Снятие, если недостаточно средств на счёте
Результат снятия: False, баланс ACC-1000: 1200.00

История операций счёта ACC-1000
[2026-09-27 02:15:45] WITHDRAW: 5000.00, Баланс: 1200.00, Статус: fail
[2026-09-27 02:15:45] WITHDRAW: 300.00, Баланс: 1200.00, Статус: success
[2026-09-27 02:15:45] DEPOSIT: 500.00, Баланс: 1500.00, Статус: success


### Наследование: `CheckingAccount` и `SavingsAccount`

In [57]:
print("Проверка типов счетов классов-наследников")

checking = CheckingAccount("Сергей Сергеев", 2000)
savings = SavingsAccount("Андрей Андреев", 2000)

print(f"Checking: {checking.account_number}, type={checking.account_type}")
print(f"Savings:  {savings.account_number}, type={savings.account_type}")

print("\nСнятие со сберегательного счёта (баланс: 2000)")

# Попытка снять 1200 (60% от баланса > 50%) -> отказ
res_fail = savings.withdraw(1200)
print(f"Снятие 1200 (>50%): успех={res_fail}, баланс={savings.get_balance():.2f}")

# Попытка снять 800 (40% от баланса <= 50%) -> успех
res_success = savings.withdraw(800)
print(f"Снятие 800 (<=50%): успех={res_success}, баланс={savings.get_balance():.2f}")

print("\nНачисление процентов (ставка 0.07 – 7%)")

savings.apply_interest(0.07)
print(f"Баланс после apply_interest(0.07): {savings.get_balance():.2f}")
print("\nИстория операций SavingsAccount")
print(savings.get_history())

Проверка типов счетов классов-наследников
Checking: ACC-1002, type=checking
Savings:  ACC-1003, type=savings

Снятие со сберегательного счёта (баланс: 2000)
Снятие 1200 (>50%): успех=False, баланс=2000.00
Снятие 800 (<=50%): успех=True, баланс=1200.00

Начисление процентов (ставка 0.07 – 7%)
Баланс после apply_interest(0.07): 1284.00

История операций SavingsAccount
[2026-09-27 02:15:45] INTEREST: 84.00, Баланс: 1284.00, Статус: success
[2026-09-27 02:15:45] WITHDRAW: 800.00, Баланс: 1200.00, Статус: success
[2026-09-27 02:15:45] WITHDRAW: 1200.00, Баланс: 2000.00, Статус: fail


### Анализ крупных транзакций

In [58]:
acc3 = Account("Алексей Алексеев", 10000)

acc3.deposit(1500)
acc3.withdraw(500)
acc3.deposit(3000)
acc3.withdraw(20000)  # fail – не должно попасть в топ успешных
acc3.deposit(200)

print("Топ-3 самых крупных успешных операций:")
print(acc3.get_largest_operations(n=3))

Топ-3 самых крупных успешных операций:
[2026-09-27 02:15:45] DEPOSIT: 3000.00, Баланс: 14000.00, Статус: success
[2026-09-27 02:15:45] DEPOSIT: 1500.00, Баланс: 11500.00, Статус: success
[2026-09-27 02:15:45] WITHDRAW: 500.00, Баланс: 11000.00, Статус: success


### Валидация и обработка ошибок

In [59]:
def test_validation(description, func, *args):
    try:
        func(*args)
        print(f"Исключение не вызвано: {description}")
    except ValueError as e:
        print(f"Поймано исключение: {description} -> '{e}'")

validation_acc = SavingsAccount("Иван Иванов", 10000)

# Проверка имени владельца
test_validation("Имя с маленькой буквы", Account, "иван Иванов")
test_validation("Одно слово вместо двух", Account, "Иван")
test_validation("Смесь языков", Account, "Иван Ivanov")
test_validation("Цифры в имени", Account, "Иван123 Иванов")

# Проверка суммы и баланса
test_validation("Отрицательный начальный баланс", Account, "Иван Иванов", -100.0)
test_validation("Отрицательный deposit", validation_acc.deposit, -50.0)
test_validation("Отрицательный withdraw", validation_acc.withdraw, -50.0)
test_validation("Отрицательная ставка процента", validation_acc.apply_interest, -0.05)
test_validation("Некорректное n в получении крупных операций", validation_acc.get_largest_operations, -2)

Поймано исключение: Имя с маленькой буквы -> 'Имя владельца счёта должно быть в формате «Имя Фамилия» с заглавных букв, кириллицей или латиницей'
Поймано исключение: Одно слово вместо двух -> 'Имя владельца счёта должно быть в формате «Имя Фамилия» с заглавных букв, кириллицей или латиницей'
Поймано исключение: Смесь языков -> 'Имя владельца счёта должно быть в формате «Имя Фамилия» с заглавных букв, кириллицей или латиницей'
Поймано исключение: Цифры в имени -> 'Имя владельца счёта должно быть в формате «Имя Фамилия» с заглавных букв, кириллицей или латиницей'
Поймано исключение: Отрицательный начальный баланс -> 'Начальный баланс не может быть отрицательным'
Поймано исключение: Отрицательный deposit -> 'Сумма пополнения должна быть положительной'
Поймано исключение: Отрицательный withdraw -> 'Сумма снятия должна быть положительной'
Поймано исключение: Отрицательная ставка процента -> 'Процентная ставка должна быть положительным числом'
Поймано исключение: Некорректное n в получении к

### Загрузка и очистка «грязных» данных

In [60]:
# Устанавливаем счётчик для соответствия номерам из файлов
Account._account_counter = 100001

# Создаём счёта (2 checking и 2 savings)
dirty_acc1 = CheckingAccount("Иван Иванов")      # ACC-100001
dirty_acc2 = SavingsAccount("Петр Петров")       # ACC-100002
dirty_acc3 = CheckingAccount("Сергей Сергеев")   # ACC-100003
dirty_acc4 = SavingsAccount("Алексей Алексеев")  # ACC-100004

print(f"Созданы счета: {dirty_acc1.account_number}, {dirty_acc2.account_number}, "
      f"{dirty_acc3.account_number}, {dirty_acc4.account_number}")

# Загружаем первые два счёта из CSV
print("\nЗагрузка из transactions_dirty.csv")
dirty_acc1.load_history("transactions_dirty.csv")
dirty_acc2.load_history("transactions_dirty.csv")

print(
    f"{dirty_acc1.account_number} ({dirty_acc1.account_type}): "
    f"загружено {len(dirty_acc1.operations_history)} операций, баланс = {dirty_acc1.get_balance():.2f}"
)
print(
    f"{dirty_acc2.account_number} ({dirty_acc2.account_type}): "
    f"загружено {len(dirty_acc2.operations_history)} операций, баланс = {dirty_acc2.get_balance():.2f}"
)

# Загружаем два счёта из JSON
print("\nЗагрузка из transactions_dirty.json")
dirty_acc3.load_history("transactions_dirty.json")
dirty_acc4.load_history("transactions_dirty.json")

print(
    f"{dirty_acc3.account_number} ({dirty_acc3.account_type}): "
    f"загружено {len(dirty_acc3.operations_history)} операций, баланс = {dirty_acc3.get_balance():.2f}"
)
print(
    f"{dirty_acc4.account_number} ({dirty_acc4.account_type}): "
    f"загружено {len(dirty_acc4.operations_history)} операций, баланс = {dirty_acc4.get_balance():.2f}"
)

# Топ операций для обоих типов счетов
print(f"\nТоп-10 операций {dirty_acc1.account_number} (CheckingAccount, CSV)")
print(dirty_acc1.get_largest_operations(10))

print(f"\nТоп-10 операций {dirty_acc2.account_number} (SavingsAccount, CSV)")
print(dirty_acc2.get_largest_operations(10))

Созданы счета: ACC-100001, ACC-100002, ACC-100003, ACC-100004

Загрузка из transactions_dirty.csv
ACC-100001 (checking): загружено 18 операций, баланс = 7959.00
ACC-100002 (savings): загружено 19 операций, баланс = 5605.79

Загрузка из transactions_dirty.json
ACC-100003 (checking): загружено 13 операций, баланс = 4634.00
ACC-100004 (savings): загружено 20 операций, баланс = 10852.20

Топ-10 операций ACC-100001 (CheckingAccount, CSV)
[2025-09-27 22:17:26] DEPOSIT: 921.00, Баланс: 2121.00, Статус: success
[2025-09-29 22:17:26] DEPOSIT: 880.00, Баланс: 4225.00, Статус: success
[2025-10-12 22:17:26] DEPOSIT: 838.00, Баланс: 7292.00, Статус: success
[2025-10-16 22:17:26] DEPOSIT: 701.00, Баланс: 8132.00, Статус: success
[2025-09-27 22:17:26] DEPOSIT: 607.00, Баланс: 2728.00, Статус: success
[2025-10-14 22:17:26] DEPOSIT: 532.00, Баланс: 7824.00, Статус: success
[2025-09-28 22:17:26] DEPOSIT: 488.00, Баланс: 3216.00, Статус: success
[2025-10-16 22:17:26] WITHDRAW: 393.00, Баланс: 7431.00, Ст